In [59]:
import re
import pandas as pd

In [60]:
df = pd.read_csv("../data/raw/batdongsan_com_vn.csv")

In [61]:
def extract_numeric(s:str|None) -> float:
    if not isinstance(s,str) or not s:
        return None

    results = re.search(r"^(\d+(?:.\d+)*,?\d*)\D?", s)
    if not results:
        return None
    else:
        num_str = results.group(1)
        num_str = num_str.replace(".","")
        num_str = num_str.replace(",",".")
        return float(num_str)
    
def extract_measuring_unit(s:str|None) -> str:
    if not isinstance(s,str) or not s:
        return None

    results = re.search(r"^\d+(?:.\d+)*,?\d*\s*(\D*)", s)
    if not results:
        return None
    else:
        return results.group(1)

In [62]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20855 entries, 0 to 20854
Data columns (total 19 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   price              20853 non-null  object 
 1   area               20853 non-null  object 
 2   n_bedrooms         10453 non-null  object 
 3   n_bathrooms        10091 non-null  object 
 4   legal              18290 non-null  object 
 5   interior           9524 non-null   object 
 6   facing_direction   8622 non-null   object 
 7   balcony_direction  4829 non-null   object 
 8   front_width        10823 non-null  object 
 9   front_road_width   10023 non-null  object 
 10  title              20852 non-null  object 
 11  description        20851 non-null  object 
 12  latitude           20846 non-null  float64
 13  longitude          20846 non-null  float64
 14  verified           20855 non-null  int64  
 15  location           20855 non-null  object 
 16  location_details   208

# Extract numerics

Next, we"ll extract numeric values and start inspecting for issues

In [63]:
df["price_val"] = df.price.apply(extract_numeric)
df["price_unit"] = df.price.apply(extract_measuring_unit)
df["area_val"] = df.area.apply(extract_numeric)
df["area_unit"] = df.area.apply(extract_measuring_unit)
df["n_bedrooms"] = df.n_bedrooms.apply(extract_numeric)         # since this overwrite the original, be careful not to run the cell twice
df["n_bathrooms"] = df.n_bathrooms.apply(extract_numeric)
df["front_width_val"] = df.front_width.apply(extract_numeric)
df["front_width_unit"] = df.front_width.apply(extract_measuring_unit)
df["front_road_width_val"] = df.front_road_width.apply(extract_numeric)
df["front_road_width_unit"] = df.front_road_width.apply(extract_measuring_unit)

df[[
    "price_val", "price_unit", "area_val", "area_unit", "n_bedrooms", "n_bathrooms", 
    "front_width_val", "front_width_unit", "front_road_width_val", "front_road_width_unit"
]].describe(include="all")

,price_val,price_unit,area_val,area_unit,n_bedrooms,n_bathrooms,front_width_val,front_width_unit,front_road_width_val,front_road_width_unit
count,18665.000000,18665,2.085300e+04,20853,10453.000000,10091.000000,10823.000000,10823,10023.000000,10023
unique,NaN,6,NaN,1,NaN,NaN,NaN,1,NaN,1
top,NaN,tỷ,NaN,m²,NaN,NaN,NaN,m,NaN,m
freq,NaN,16250,NaN,20853,NaN,NaN,NaN,10823,NaN,10023
mean,48.201716,NaN,1.154800e+04,NaN,4.015594,3.744228,37.061121,NaN,14.893263,NaN
std,331.912772,NaN,1.085759e+06,NaN,7.196456,6.514958,1945.315351,NaN,14.321417,NaN
min,1.000000,NaN,4.800000e+00,NaN,1.000000,1.000000,1.000000,NaN,1.000000,NaN
25%,3.900000,NaN,6.210000e+01,NaN,2.000000,1.000000,5.000000,NaN,7.000000,NaN
50%,9.150000,NaN,1.000000e+02,NaN,3.000000,2.000000,6.000000,NaN,12.000000,NaN
75%,28.000000,NaN,1.850000e+02,NaN,4.000000,4.000000,10.000000,NaN,19.000000,NaN


From this table, there"s 2 noticeable problems:
- The non-null count drops <- "Thỏa thuận" prices got converted None
- 6 unique, non-null `price_unit` -> Price unit inconsistency  

For the first problem, we'll analyze the observations with "Thỏa thuận" prices 
separately and use the rest for the regression model.  
For the second problem, we just need to standardize the units 

# Extract city

In [66]:
def extract_location_detail(addr: str, level:int) -> str|None:
    '''
    level:int - the level of details to extract from the address. 1 -> Province; 2 -> District
    '''
    if not isinstance(addr, str) or not addr.strip():
        return None
    parts = [p.strip() for p in addr.split(",") if p.strip() != ""]
    if len(parts) >= level:
        result = parts[-level].replace('.','')
        return result
    return None

In [67]:
df['city_province'] = df.location.apply(extract_location_detail, level = 1)
df['district'] = df.location.apply(extract_location_detail, level = 2)
df[['city_province', 'district', 'location']].sample(5)

,city_province,district,location
849,Hồ Chí Minh,Quận 12,"Quận 12, Hồ Chí Minh"
1134,Bà Rịa Vũng Tàu,Xuyên Mộc,"Xuyên Mộc, Bà Rịa Vũng Tàu"
20534,Bình Dương,Thủ Dầu Một,"Thủ Dầu Một, Bình Dương"
415,Quảng Ninh,Hạ Long,"Hạ Long, Quảng Ninh"
20657,Hải Phòng,Kiến An,"Kiến An, Hải Phòng"


# Select columns for analysis

In [43]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20855 entries, 0 to 20854
Data columns (total 27 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   price                  20853 non-null  object 
 1   area                   20853 non-null  object 
 2   n_bedrooms             10453 non-null  float64
 3   n_bathrooms            10091 non-null  float64
 4   legal                  18290 non-null  object 
 5   interior               9524 non-null   object 
 6   facing_direction       8622 non-null   object 
 7   balcony_direction      4829 non-null   object 
 8   front_width            10823 non-null  object 
 9   front_road_width       10023 non-null  object 
 10  title                  20852 non-null  object 
 11  description            20851 non-null  object 
 12  latitude               20846 non-null  float64
 13  longitude              20846 non-null  float64
 14  verified               20855 non-null  int64  
 15  lo

In [68]:
cols_for_analysis = [
    # highly relevant for analysis
    'price_val', 'price_unit', 'area_val', 'area_unit', 'n_bedrooms', 'n_bathrooms', 'front_width_val', 'front_width_unit', 'front_road_width_val', 'front_road_width_unit',
    'legal', 'facing_direction', 'balcony_direction', 'city_province', 'district', 'property_type',

    # might be useful, just not now
    'latitude', 'longitude', 'title', 'description', 'location_details', 'date_of_posting', 'interior', 'verified'
]
df_pruned = df[cols_for_analysis]
df_pruned = df_pruned.rename({
    'price_val': 'price',
    'area_val':'area',
    'front_width_val':'front_width',
    'front_road_width_val':'front_road_width',
    'legal':'legal_docs',
    'location_details':'address',
}, axis=1)
df_pruned.sample(3)

,price,price_unit,area,area_unit,n_bedrooms,n_bathrooms,front_width,front_width_unit,front_road_width,front_road_width_unit,...,district,property_type,latitude,longitude,title,description,address,date_of_posting,interior,verified
6058,69.0,tỷ,122.0,m²,NaN,NaN,NaN,None,NaN,None,...,Hai Bà Trưng,Nhà mặt phố,20.998051,105.850369,"Tôi là chủ nhà, số nhà A980 mặt phố Bạch Mai, ...","Tôi là chủ nhà, số nhà A980 mặt phố Bạch Mai, ...","Phố Bạch Mai, Phường Bạch Mai, Hai Bà Trưng, ...",03/10/2025,Sổ đỏ chính chủ,0
16597,14.4,tỷ,64.0,m²,6.0,4.0,10.0,m,20.0,m,...,Bình Tân,"Shophouse, nhà phố thương mại",10.732635,106.620926,Nhà phố Sholi shophouse - Mặt tiền đường An Dư...,"Shophouse tại đường An Dương Vương, Phường An ...","Dự án The Sholi Bình Tân, Đường An Dương Vương...",13/09/2025,Cơ bản,0
13102,13.5,tỷ,73.0,m²,2.0,2.0,NaN,None,NaN,None,...,Quận 1,Căn hộ chung cư,10.788800,106.699013,Bán Căn 2PN The Marq Quận 1. Gía 13.5 tỷ| Danh...,Sẵn dòng tiền thuê R.Y 3.8% /năm\r\nBán Căn Hộ...,"The Marq, 29B, Đường Nguyễn Đình Chiểu, Phường...",28/09/2025,Đầy đủ,0


In [69]:
df_pruned.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20855 entries, 0 to 20854
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   price                  18665 non-null  float64
 1   price_unit             18665 non-null  object 
 2   area                   20853 non-null  float64
 3   area_unit              20853 non-null  object 
 4   n_bedrooms             10453 non-null  float64
 5   n_bathrooms            10091 non-null  float64
 6   front_width            10823 non-null  float64
 7   front_width_unit       10823 non-null  object 
 8   front_road_width       10023 non-null  float64
 9   front_road_width_unit  10023 non-null  object 
 10  legal_docs             18290 non-null  object 
 11  facing_direction       8622 non-null   object 
 12  balcony_direction      4829 non-null   object 
 13  city_province          20855 non-null  object 
 14  district               20855 non-null  object 
 15  pr

In [70]:
df_pruned.to_csv('../data/interim/batdongsan_com_vn(1).csv')

In [71]:
df_pruned.area_unit.unique()

array(['m²', None], dtype=object)

In [58]:
df_pruned.front_width_unit.unique()

array([None, 'm'], dtype=object)